# Train the change detector on a Colab GPU (several seeds)

This notebook runs **notebook 1** (data) once and **notebook 2** (training) once per seed, unchanged
except for the four settings below, on Colab's GPU. Both notebooks are stored inside this file, so
nothing has to be uploaded: the data are downloaded from the swisstopo API, as on your own computer.

**How to run:** *Runtime → Change runtime type → T4 GPU*, then *Runtime → Run all*. At the end, a zip
file with the results is downloaded (`colab_runs.zip`). Unzip it into the project root (the folder
with `README.md`); notebooks 3 and 4 then use the best seed.

Why several seeds: the same settings give slightly different models depending on the random start
(weights, pairs, augmentation). Page 8 asks to compare runs over a few seeds before drawing conclusions,
and to **choose by the validation score**, never by the test score.

In [ ]:
SEEDS = [0, 1, 2]
MAX_EPOCHS = 80
PATIENCE = 15
PAIRS_PER_EPOCH = 512

In [ ]:
import base64
import gzip
import json
import re
import shutil
import subprocess
import sys
import time
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
WORKDIR = Path("/content/blatten") if IN_COLAB else Path.cwd()
WORKDIR.mkdir(parents=True, exist_ok=True)
(WORKDIR / "notebooks").mkdir(exist_ok=True)

import torch

print("PyTorch", torch.__version__)
if IN_COLAB and not torch.cuda.is_available():
    raise SystemExit("No GPU found: choose Runtime → Change runtime type → T4 GPU, then Runtime → Run all.")
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none (CPU)")

In [ ]:
if IN_COLAB:  # everything else (pandas, geopandas, matplotlib, scikit-image, ...) is already on Colab
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "torchgeo==0.10.1", "lightning",
                    "pystac-client", "rasterio", "fsspec", "aiohttp", "nbclient"], check=True)

import nbformat
import pandas as pd
from nbclient import NotebookClient

## The two project notebooks, stored in this file

In [ ]:
NOTEBOOKS = {'01_data_swissimage_tlm_rapidmapping.ipynb': 'H4sIAAAAAAAC/7V923bbRrbge39FLWX1NKhQMMCbSGZxZsm27ONuJ/aRlHilJS0KJIokWiCAAKAlxtF5nA+Yl/mTOe9z/mS+ZPalCiiQoGwqtrqTSEDVrtuufd8bn/4iDqYyDLODobj8ixCf4B9+Ms7XiYSnB0svvfXju+igSe8CHx/25bHT6w989XApc8/3cg9efXrgR1m8SqdSgYW/vxOu+L//KV5Cq6E4//Dm/PzNjyevT0XLabWbIrsLsuzi7Y/tl8KLfHHmJYEvfvSSJIjmV2pocVD+dnj4Po3/Jaf58PBQTFZB6ENDMV140VyKIBLPQy/PZSSst//1f/JsupBR7oUN4c1ymYp8AW2m8iiNp7fC++iFXgQtRDwTrT4MusY5de26YS8WQSaiOJeTOL4VSSoTL5WZkB9lus4XOAWErabhyxwmGKciktLPmmIWh2F8VzSC/cnk3zKB+yaSIJFhEMlyKOut0+J33vS3VZAFeRBH4v/9z/8l3jodsfSy24y2aroIkkw974nfVl4Y5GuAHuVpHFKLLIFHqonriMSDubWgxRIm700Cbr+Q09usMaxbtGuLc+ml0wXNm04qj5NYnF+cvBBTmGEYz1eShkql54vDw43jhSOawTZkwb2A7fIyPCAEZZ6Nsd0tGC+HdksRR+GaWhYnHHprOEA4qcPDEmUAPq1zNclkLoJc5DH1gpMN5VpYoTeRYdYwhmjb4gyniq0ODyvYBsDiNF/EySLO4wyHcgcFUggLzvwIjjvKGzRm2zHfxVmuX+KCFRIa43ZgXC8DHAx+lzT4LI7zJA0iOB+YNJ9qzLsTLPGg5mngN/l48OlSfAwyOONw3eTTX+Xc8Bn2ZWQoh4N/7oJ8ITzcrsTLAy+E5eWpBwfwDLcnAARDtHomcpnljCnGdLu2eKFG9nKaVH4XVy8noijcADjXAqNCKSx6CvOO4WbA7PC3o1TOgwwGxxEbO66X5Nkh4OKaWTdOa0yPx3yzxvpm2UGyjiY3fBQ4PZhTGEx5Tbq/caVunPZ4woeiQS29pIBC6EYoPAtCWNRdGhARWchU2gcI4xr+RfTtS+mk1+92J+1pZw86+Z04scWbZQJYyDcckDqHDcnqduwkDIv3dAjqakXyPhc4vc/Pexr7Uk1P3ktAKNi8MVCnKIe30SoMzfV0Pdc7nh13a9bDEypAFI+ga5ysJja/keMgSlYI+gAuTe/IdY9azkXLGXbaQyC6re5x23X+qZanu2a5l68ye7LK1o90dFq7OgZ+KGs7th270+n0ut2yY7bAXdOTBQofrnd3bXd7g3/SDosH/Lc603iVwxqJp17Xn3JA5yuyBexVWB6nepzLZYIouP0iWG4/vPPSqIof6sXvwQaUWRovgf7nizCYCNXoPfxZh1nq9SzLEjndejyXcQK4CbQc/j9P/K0GSy9PwjiHgexkjb9hwyTMtxpGq2WyxpdRsvWuHKJmhGQNpzsdT8MACO7W25TJbLy5+HUCcoNe+0XqRRlQ6qVMN9rp7raE+WW6/ZnMvCVQGFMmqTafSUA4FAmq0/hd7uoAQ6PEojYN/9jVMteT1a3x9ThOg3kQ7eoD0gaQpGI2H+jPjcbZLTEQ2yTPukOy8DI5nqZxlgFJSFMZ0us6fNFYaAPKweD6T+vqIJhHcSqvDpooLMh5nK5HP2cy/cAtGnXAvhOjR37Ey5OLE3F+enHx5qfX54825Z8SMMos45/P3oqRuDpY5HmSDZ89QwpmA0bbnr8MInu6eAYc7hli17OP7pUxrRdn59Tx9P3562HL6faugMV+J85REhFvfxl0m3CEILoYV/HsFLs4dhcbepEXrjMQIaFJHK54pyPVR1iG2ARtHNsVS5CMJchcuUzE89qdKvuMfz09OaPpocxlThuEpPHbk+enb8dnp29PT85PdavukdPmJYSAMTCrEKQzkLYkoBnLJyTN/FCR1KAZyCDSJ/yjRjiemIXBfJFXB33/7vxic8zekdPiMWdBagxaiuY0ZlOscAxiyUgCInjshSvGzlTOgCVHU+OunJ28f/Ny/P6Mx0m95RGvr3vkDnIXuQP88LhV+YXnvUvI2xoAVrQ1QtvJnb4z+MwIu0TFevw/0YIyopWw7gExm2JN/75fevf4u3ffsMWHRQzy1scgDOEOAymOUY6MI8myHO8fqBK9Icw5kndAgj3YtgNT/jo7ff3m3U+Ipp8qoiP0gG32ZXSFvPzqgCDi71cHJI3hjb46mICk4Gf42Gr1Wp1W12kK120fd/EXeNKFXaEn/U7XaTw0N4e4DRJQffYYogTo6iH6atCBUztE6K0Ad1O4fMk+47ga6rHD4/Ta/KQDp103zh0w23R7BMDbWvgDBb/fV/CP9eYNerXrmHkzGKB+FSC+1wzSRqykKbtqkHa7rZ6AnFU3iJKPt4cgbK1byHFfQXQcNQZIYmrUwcZCHuqx3VD/4wmq9QKl04z1DrhGoH4iFUjjeEaXAshBqaSVgJ7//ObtS+AJ44tf3xPdvbw6eC0nnlz5kmf+bjaTEQCuPv2HBygYqj/OvVvY4602b5biubeqNEnFxSpdXh1c1y3qxb+9eT8+f/NPJEitbg/JQhLcgx4Kf7utvlgKWBryhaVBM89O3vw0Pr84e/PyVDWDbqhDhYqWsLYfRFngM4Eu1CVNAkpop7+cvC2BqTlEsYZHKqqhA5IOhWhUAPjH6en78emP7y9+Hb86O3lxAVSCeFkLAWULVDdAPeYpoZ4JYi9AWZfaOh0fsLJbmZD8F8k5jPXRnOT56elLBPqlvH//n3qM+0lph5kIYUKIU1pfzJ79QHaXZ7QjSpR/JjKiplKEcA9CsUqQS5AFCrEyr9DTd++QR6BgbU/vfKtho5Uogp2YmQ8jbylhqUiX9cjAPQBBpNHK5D8fACjBfgZ9aIapd2fy+fdn716cnp/ThlYbwlSnMgOGerWBoedbTdmEYDR79eb1zyzFFA31psyCOQq7ZmtEqlkc+nA5YE8vYdZNUcyrKWjMplAwr4dV6sMd7eWtH6QWb1o2ukhXsinkPQin4/iW/qzlmXk8vptn/Q5M1JDrbZKSp2lmgfDW1LJbp93q4VX2wjtvnY3v15tglVkLYFXUDPsF/ydOZGRpWdLoRqYcEHnPQJaOo2yIY3zCgx6K9LKgptd09fBxU9A2KQ5sB6D4ZVbjofE0YwMwrG6n3z7+cmPDH+KcjQfiD/ELiFcS/vthsRZ/lEv64+joqPjHeCxu4ABvoD3RMPjvCdo9Q5mTycwR//W/gX7BCzIBgQA3QerQouetjiKFcABIl47oLyIkQFZQX2ECicQljnJ5n7Oxq6B1cCLrTICulQVobULiBRJskseJePH+Z1ugHQmtpIZALe9Rc5MoVWZsx8TrR3O3RXVdW+IyrlJJy/AbAmeLoliuQID1ZTZNg4ksoS7jJV53LzPseCiDIfpRA9Qj2JAsZBSv5gsUyMQbNN2QcsAS8eEh2ZVNA7ey7ijD5gJZIgrjSL7XyCPZmB7Y0kYz6B28XyVoitsU1o2NUdKpRSZGaAhDkLrxAmZ0UswlWU3CIFvAe0M50F2lPbcJLmxQU3chSD+S2RhfNUD5i3DBeKIercPLi9lvHoG6ELjxhWj7h3hF0mhT/INERkKJt6Vop+yaOLFl7MuwCeou8mjF4lAnBzx6pWUo4nX4zC6dBWQs/SBBFpFRGsASyBcBvzTouPA4WC3B+6uxcYhbzTqdTD/qlzALkOBDVmYqyzO40Hfiub3pC4EFe9SdzOugdDyu9eHI5eHAgbt088TtUrwI45V/9C7Jg2XwO7x8LeOLN69eiZyMmyR9uIj9J6xnOK2muAMMLkz9Va3ZsPTrBTIgZvJwTdMMuaH0oDea+UuHQGHGZ3MEjoIuEdh4OfVgP6uDse1VowkBhuMlqeVjIO+IaPhyCtcK8ZoG0JKUYBsJ0htWHEW2hKsigJUQWaK54awNbn1TNcXwv63GDSBR5gXTzFgpcCVYIcCFubGHgUct1Ho21hMWHR56bPo+PCzHSgv7EVwanAv3Z/IX8FBLicLtTIF39WvEVEVLG1/Zpnvs9Lo9fyC/iU2XzKTHbnewr02XOvZAx9vPptsZum2757RafWdfm67q6vQ77pNsur6ciZQY/xgRwWItCQliNgJe2Rhuqlr4vzOZr9JIFMY9RB8fXTZweRaS6KtChgqW0R301HA2g6qCr7cYgECjlLdKYxqzqQccgUgLbSwL+4kjAtUAwQ8W0mjqd2t+tzbeVWGmvDLDSllYMXA2qeStaTSro9eRu/I33GOmUGMPqHSeWdMYeOEUsbMp9I4juUdT+egnkNiBH/qZ+i1bzWbB/ejqwM6DGTDd2iN5FShv0iKVM/IAIg0hGshjirtFDFTrFliwhPH4wt8w7Bvli6rQxBue2M2fP6gwjkCzBn7lQAMt8ZamYcNO1Njq6FJHd1fHwqhU7aj4wUjTf5sfGPuejS7LP67hFCbx/eiynGnTGPzaOBz9Cx8Q/LMxMknDMDDwttziUbWEvNGSDwrUfc/GX+lyYEvkawwFn8B5Ie9Qz2w+Sw0RlTNoYOOB4nlafJyN61qkpk5NHvfzCIs8aszsxKIuJaYCVRmjQ6YeE3+kPtB/CRqiePHudVaynio5ID8hcBy0jJy9fl7weuX+JRkI/fzoJa7FQli+notNuhbsycac8IcVnBmaOafAwKQPHEH3e4ALtd1D7ZhutIFcRELp6Ao2TLrVgj3oCzwt2rLrTU0xFVk6JWmVYQzRKPHW6T7DGAfg1lkcDUu/uZiQMp+iXqok2FWGfBjtFSShb88cEQTkhsspq+o4JA+YTm1A+DilC55cDtvX16jIg4KXooaNYv48lVJbFUGtAp2vudlvY/+X5CYvnTwj9gZZankaZUYbXIV+UULFqPRP2Z52vEcxMvMR3EIga8AC4bbCXQRZr33d+Oyebm8KLSIE8mdt9CYUg1mzMwnoiOUBGo2chvjvpnWHkShGIQ8a+8E0t/w0gMkCUX59EcxmuGnEEEYMKVt4ibxsXWv+UHmM9ITEmhEoHj5KPwBmBfvbBzDbkzd+pmk2IotAseMjgweXm4ZBDbDDGcCFqwwkjMyBKA/6mzYDocMtqqisMR9x4Q47Hx6qDWigvO5nec02w1Mbww+kRctt1Ldg1TMhIgy7aT2GgI3NMyguMtIN37jGcKWrW/8g7quPXHiUALv45DqOOOSjt1FwBZphu7OHv/KjrUF30oK638gmTKNSY1zgJ8N8DGJgiqIKchAb/2UipGFfIQmpxsgyrGE2zYKTVGWM7d2/OgBmVIRC2eVcj/w4cR3ceB75srSWX9dgZMEM4SQ+bfjyHo4cF/7/bPu52zpquzhGIc+MQVUYo0eSBZuNkXadPZmm8LxDJLm49MaDqY9ujcyK0Na5bh7VJcK9Rjlym/PVbEtpIARBEqZVgtuawJinrMS3TcMbPHuJNlo47U8GXoCYStgytJ3Zg8DpP82+dtwb9AaTlvxy+xraigiHSFhkz7i4qRzWDeAayk6s/QHvmi6RQPqraa6URNxzEEGzmAw+nkCrGuqBG2FUxrkpDT9FzzGwOrIL/P396esjTc3g0a8vJi/SH8TrlydvATge1BrValAcMw40AzkDJYm7hYzoHIGrGGh1twhUQCCytFXK0i4Az9mbwJzXuoHRm4KIEXAwIEQ3DVTtZ0GKtw2WFGHUW7QCBR8eq1WxsUX60ufQqx22kxe26TcqbGRlSN1OqsI9AmUliGfoTaZNQuSJaAEwIaQA6BtI4My0Hcxqg8YOuwJtz5EUEhlng6NuDmC9crx/vnlP0WRspLyTE0FWopRVhGyVcJzXv11cvAe+gWGjqfxthaYp2p47iXYzgYwEDQBhcCvR1hlPYX4EFhmOYczgcB1WQuQ9cLRpXppfZji9pVxOyEDCWuWdRF4VFdt3VGyfirW02HLb6Tvix+fQyzC4K2MLuqNysvPMBG+P2pHZCvQmhgKbolARY5vi1EsDDGGchmjK83XAZiXMAJfgw7bn0v/KBo++609nA2fyDQweyobQbR+7+xk8dEenc7yfwaM/dPv2YNABerKvwUN1Rc/2kwwev5y8fXv66/j5u59/eon+IXQGt7vKO9/TzmD9pNOBXxqf15bycDnW2JhZ6uIV4m9lzHqtSQUPF8HKO+kDIize+CIEBTDuVsqEb8yjSrsWYdArdvKB2VZl4uNPCuqDPU9u509Ut5TANE98m1gpXvlCoKxRl2E6GPlpPaar14kuMPW2T6wYVfFLzYPpcbkSaNB4RAPXgxtK+H4q99UByJeJ/XuQAIveJbPw6eJBbgSFk4fjEw69rYg+KiwWcjsTT6WAAhzkWPH0dpwFv8uR2wNJt3V42HIa3IKkd1bNmzrU0v5nkLzCM+Ln1ETxBZDEr+o1Eh3taV8o0rh+SYbmOF3zIPkyqcEN3F8m5hQRwaOQqkoGkzp0UpgHmtbp85PTn1+ejl+9e3fx/uzNTxfjD6fnF6ANUDgaAh3uVp+Y/bCH24LJof0PDs9gvAfie37LsGykQN5UQqMr/CG15Bkess0SrB3GdzLdwirzhy0ZapV0AAybdgi00iYxSYvmRqrWBODvVrDMH47CBd08WeM5xJN/WQQQegLhKs99G0px3WE3qpfU3BtzaxDDcf1kJGMKswG3gGnn8aMX3lDhSIAvOoIQbxA5Vu+Aw5YanrAeEY8bu9S2Av7jNByIBt60ChXfcqk2Ku3HGA5X28mMIKzlHcUmwOUyqP2nrRG1mgNjVPanSd7YT/DYnqbZQ41WgUZ6w8lKTn1oDoT07PSXN+egVI7/DqTRxvhEyUIHUDc8PTJvNLYAcud3z/9++o+Lk7OL+r5ZDm3n0PtpuorTmcyOW/19dJW7WKUBYfB2XhrKgJGiBzgO13O0M0xASVhqF3htwtCRODy8QPw6PERXrt6jG2FVvNXkVOXYUSBl5Osuz6Xq6iZZONeOd82wTSyFKcPks9LFTfpjukZJFIYJ0Gi/RnHaj8nvrtyO2m0tCjmDFRtQZNhnXPq57eoCX4TAzUS6Ak0L13l18KO3gj2bY6w0XEkr9DCw+w6mn3F+Clx1mc7Je2LQDVTScD4c2obt7uJViGKyt+mFFezXhp0BVSkDGo9BBjkOA7dcxxesommIkQA0MUBmvXNrGslPY5S3v7I87Q0GA8/x2t9AnlYiar/n9vaTp1XH40G3s6883XJsp9N1nc7+8jR17bht50nytIoqITp4WUMk4OpEVjXGsXFNbMsUZxiKSVLx18vilycBLcjsW7qAJflUVFWl2D2gSF2S2R9EGeesb1ilA82oUSW6ZiTAHKRHdIUlYQ6SwgTzSDILHpNEZrmtpjhubC3dxmaWdz9CjxYZ3EcU1jq9RcqNWZZsXzYZOhsxyMZYhARTXK03GbL9tKkjefXjmNR1fsHht/qNYXtVQbP6DZlnH/6MnXKnt3Dbrlbt6N3bnu8j/weFAPfzDERMWAHIF6bXkKEqp68CfqTGA2EkHL3yQiST4d0Idl9OR2rnLovxdZDb9aY0hzPAECCgq6XxEdi29Un1/Rt1/dv1QwM3z/AWgzTZcxr6MHePiDm2gJOIHANjcBgY9I5xHuQhjfy4nGASZ+UIVrmTQg20Dfo+DJZWRTW9dGA21Set6+2O6+2O7lbHdk1HD1UVlE0kpvxWpRY43Bw51hgUI6A61sY7FAjhDlk6sBMFVMcdq0WOOQk0ieZ4CH4SjNyuswlhEd9ZTxRMuj3Zctyut1dG5Et7I6fCTNG13MFmLm5jp+kPdd6/ZbUZGpUESg55KvRmcVNVmhP5O+wVCEhhIOcroAPRDdm8jKxpTihh0E0RAdkRPoYtGgF0CN9L8DolafwxQNOrJw5/WwXTWzMLWVg3v/lxctP4gXqpdSrIhiANAhyNQKk0FZizAA2aaM5lY7wtnsekTkmV76oiunQkHMVvkav58FDeexT0pMMbywE5egn9z5iBzbLSZjQbRz+SsRFD+QGuFrFUvBNnAYFE43OGOwe2UMzT0WTNoaEmncYc4yCLMbW/SGb/2hFRg5bv+oNZ65sINCQj9J2+u69AQx2P+11nX4Gm3bedfn/gHu8v0FDXY7d//CSBhrjfmBkSWsqYs10aSSbX9QwrxetZ6/ZD9CA23yST0hgD7tiGQPHmls5rw7wvJGBFXhhy4yO8RkfpfHKkrgG7kR73FJs/Bvi2Y4AHLZXgb4G+3svNePA5IoNDmpuq7HVqJ64Lh6AKXtkRJbPYiK1Am9CiNMMpaMB062NgSvchHQT5i3FZwPIABx+aXLACbf3lc+3nxLfkD/tEBhIaHt5zNsS23c7Ag0sabIdTsbonG+5EBqImq/P/1fk8jYe1OpPJoOM5e/GwUyC62mLJhR5AZ+vUcipdIwKbkt2rjCNXZLbIgOTwn/nKA2k0l0U4LrcFpkhlIThXpxru0KyEpepsVOU3ophlSggAAf/GucFoNJCfWXttihuXnqjV2OId2stVmCrHxk0BAgwJiFtkK3nmJdZavjImFKZc68bDzY8xqttnWfOmoTkG2uZ1ElhG3C+kSifoUaO8olpP4auyEgfq/xRVQXUs6BCGtJ1Fwqch/Fk65F8pLmWVCUOpqbGJWSrJtehHggHr34eHIP8Cs9wKIm8qL13MEFUe7HbCq5moJNpfme31Zl5v1nW+DdtDTjJwOx1nX7bHHVvu3n6x9sDuttqD/uAJbA+7to5b7ScHAhOajRHFLG05a5bHqAKCHo/+KyvFFLa3IvqPbj1IVTdVkDfCcjbuqmvc1EatL6smZqkKVXk6ZrXxgDqUC97b6q8t60FpaPeSjSA3HMrmyC7aH5t08wb/XjSsgiKaNiprK1iWBRfI5ZBBSpLRG4ZZ/Zhdv27wZtMERpvTGNURxVr+Txqwsx1qJraoVqM25M3mdByLY9bcGkAq8gzDlb95aBnuI+5brbUfXz5u6Gcmti2eUbRkacgYbh9eGSBkXBM23jR3BBNtcXXsU6O97woQ2o57GrqtrFT2Ffsa6pg2Y5o6sq1rt2YPf61AxVba0ra1FHpRsx5T/N29LNPzsSm11NjlVIM63qQ5s7UNt1FZL024iONTi32i+0FKdyLd430kpO/Ev1erlw1VqvK60JkDzp156/TqAwko4AouIdzvjKvlIEudYUC/p3wHcEEmIYaGF5r4Ol4JkK1l6qFgsIhXmYf6CuqsOuGQAqpsk3B+AOEGeTAqteL3OKZsRfZhKFNeIbgZvjg/9e4iKvAVJ1+beU+6vbY3+DZGeOKHPRAN9mbe1LHVae/LvDsdu9vtdo6fENRCXTsudn1yFg9oGKDLWY8wZ8o7K+LsKTDPSuM7KnqGZAwDztHzk6ZoqwGyiFZpTLX9Uh5ccN4sne4ODYkSewm3BGOt0WlNHmgLuB0wqSP3C6JtEHXHgFal25W5AhlRO47zaNYSrPcZrBaFcYzix0wZcYM9b3SqH+sBvq5p50sQ1jODNKFBqHZDyq3EsJLEjuLod5nGNLtNhgXtMPAcCCHuhvQDIGDYv9FobjxFeJtm6dQpO2M8mpWiQxzFrmfPME4edpKyJjj02blWbzfBTDfBTB8D4+4Co46VdtRKoRNM73tu2VRPpxiWXjx9/IiVC4VSLCpOFEwAQIu+dqX0QRBoNTZCqNG9oLn577AkBGRjEHxTM/dN9IBLozV0vD/1vLzxyFHX42N9R+/eDpZki4axLkswNe2Qo8BFN4GZHZpc1iEbXTp291o5GjDmfy2xZmfFb5SNHLu/PULFw8BCCHBcbfLiEUsHx/DxMOvvBbDqOufElkyDThW8+1cH8Wz2NdwAxCXHv00rHoDW1/QA9HqO7E2n3n5h1HwSKMXjSWSqHgcbQ9AiAFo26PTZGkjoEpTmqZCzGYVVo96NUQdYKYD96ynQ2tuIS7Eqy7dRP64ax4wZH+mOKAfDj0TBrZmqJ1owfND0LQqyxQRCnDabMkK2/hsGfq56wL1xNYeHyuZhHjYKQ1i0EgSXNWZhzhch1gAIkepSCTGUddSvtjgpcqJV/ANGOuJiYZeUQOiFMewACy3Sn8tKhMM5ZU5jVjn7wRYrtZdLj0J2gb4HHwNf1SLNYkFgON+aa/iAxOmJGdacouEa1FunYhfBHFmt0aZwT7DQFsUBxmdyDVJVIQhnnDUpeOIuxYWoUTn2mEMl7kBfwz4pS3qcwZ9yirqB0DKdaYjilfuDyKZ0mnAqGIoxUU4SJdAFlC6/hPXvCkx/hWVTsSgOV981fIdk8muKXfLrKU5Me4IzLu6KGj/W7sG6GfAfKs/CS2SI0A6z4TFYHrQ/Wh71r4jMiE8cbE9FNsyCgUYFIMQOKgO7oKrGKlZF2/U4k7AQiQlQUdVCLTKU3i3m0nE5i60iRcaoqqpDU5UPWoKQpOsSWRj6BD3dVr+BKD3HsPWirAP8OyUTXMBpTwqXqK5GwTR+MAobGa5TVeFIjakigdSw+sxqCxkR3oNyFJIoJ1GtAzUJBRwqchBPb4doW6TgYc90Y4AQtBQt569qmstKFaQiqokXRikeGdpwuSgIVwaiqtKIwVcHX9vL1ZFuT06+jbkPhPDecbc/6O+rMXDHXmewr8bQde12r9123f01Buraag2elvefRnOWUfm4bRCrvVWYj+G5heWtGqalHSgL+cIur/9MFAhffSPoQ4dAbEWZq5YjIxZlW5dA6SiIVnIzaJqu4UhUypJVYapQGS5bZRQc246W94MUQHHNqWcqkqLSyNKtUBAhmRHj1/YqBrUFhsS9p0D5TpyH5LTQikz8UeqqLrjl7IknQyTm+6j8KSWPEC3itGyjBNBnFb4dwvJuFRDxBoRYRBeKRbIcTjJWZRyORFmD7nvhNtWB7goMp/zmONyCxvUo9gTGS6SdG6mys6h8YfWIu2YJyvj1kdBv9maNRKHfMuQR/+eRjspQvCH0D3HPvjcnATMb4tKNh9ePTGcmrO0sZ7gOTkMZzzDl2XZcSknPboFPljXyQHDIKPFcxac+bnCuv5jmj2ZR4xnmeiHx4PWqmTy6ipq+sAhmadFckTNaTU0lwOGfnTnuyZhSxguFafwJj8fp+vAbngn+tmkYMH9K7wOixue8Dxu9tC2egwENbFB55saT0lPAFwLRblzWzlB4qNPC6+z/W5kM24b8TdLH9uBPapuURXsvS7/58/mE8h3bU7gqdnsm9licIsjfam31Hg3zR3FgG5WYyLc+YcxFQAVV1WRwFkzj8SGzZaMOKv1CWfWblwffbj18qJXzmRyMROLb+D2UVymMYqmZNTaaYS7ANPtoac55pR7DM9wrKuqwhXLZagmq9xqrtlBjFOWTydoqFtKwvfl8I6+dmo4svSO4SDr87QgZ3h9qq7cKy7ssJ74nUox1xb72v+IAmBpoudK3MplbaQN+mttssLT1IMCafS1gz4ZkYbNmSGAxdwio0xbIAgArLuNkmn8BXK5qxL6QmSKeTdGqQN/eYDuMp5eXlSLCRbVfjgm+vn6akaTvzfoDt72nkaRQWDjiIyjqoJEaLlM0x+qCRrqsWKQsAIQZHA4xpdyGYDmhT+NISq0r117E3XVM2QfzymWZt4E8LpRHSEKE9uwhY8GyhKiUT7jON4oRoLtj+h6IH4f8UY7DWsX6RCh1KuNKjqZ/hlC3icritPxeyWYQTI7jUp6ILjZOuvHwKycXH8se5h18E62KFJWe0+nsq1VRx26/1dtbq2rZ7U6n039C7CB1bbs42SdoVeqUFQW7tJiQbaseDfHfhHq5Lc+QHNZqXNsMzeo1FRKNcd1ytKGePWYt7xnW8lanKfqbxvKgWYg0eKei1ZI8ihYPbeO7TWm5hlmWZF6rVp+VCR7REHAtl1j651pbyh/3FzWeOL/H2Prnpuea0yum5GK4/9JLRljKxlsjSf24DCIMvvgIYvfI3bKDFystLfEFS69NC2APByXPar/G8KsZ1/+Efbw/6fe6srdfdOFrGz8ZUHyjSQe8cRJb+WWm/1EJm661vLLdEM1fle+YkUtb0gcGFmR8VhHsW18Rw+zjIqCcaXDiBalh6EIGUA44UZ9toiTVr2ziak8nPb/b8r8NMW4hMW6323sTY+ro9PaMaBtgTcyW47a6rX2JMXV1B4N+64lOcUIjSo7aCPAnmfjxCJfDw08zjGbDjItPtw/Y4yNXB4C7zOaGIsq3KIFtxBRxcC97FjmKd7jhoRdGVDhVZQgU5hfgNuzdJqT5pNTjW6S6m+BwWBaReA41EIvWlYbDR4Omhy0MPuK56I+kaVO2mYQxrNTYUjHQZUCSuxmQNIlBntZL5D566824+GtgmvVvMaz9uia+6BeCiX4F9MzUfAJuSPioZltO45G5fgHfdateauS7x3VOaqs8rYbprG5uH1xj+IgT+XOO4ypi7+czbn/OZ8wB6sx0d7h/zeJG39wTrL+Uxzfpm7mD/d6k0zqeTL6c3R0evqh+TRATsD+AQgEi/pKDuEABWQQzjC/P76RUwSdExGh76btWwviilfqoCOU9kfLgB8YXx6huOTFD1o8W3lIVujeKhTeplpK+xL6cpIFy1IFmtArJU4ehqax2QTNVfP4cJ4rBaKH5aS701nJdaFTkQJtbYk1LlR3FbJTmSfXstftNVcv/QVDyecp7kKmM8imsDfYG1XtOXGX/b4XOYvRajIFs6Am11JdC+SsPXzvHauL1OrNJ+1tUnWZG2e4O+sf7sWbd8bjnPoE1t12n0xo8iTW3nX6/+/Sq0/NUrol81VeRNSqKg+wmuRgq6uOIkACew9M2r0R9WJYK/AJ6CmoBnDvWfQljL2+3yM2CxF5x0scDk74zr84QK7Xn9OVQ5TRQVgRdncyMGqjkKFo6BE25XsqUxMZmoTIq7DmG/xd+My7TbeYOVTPCHXJXYFzXf5Qh64dU5qqvPg7VcZ1KdqxaxUhYHKuF3BNhqEg0+lV8LzCwTodz4XMcTDWhX1WT2j0sc0FGfPAFi7uskc6uL3lKZkwUBj7VuCALGaZWZECTVo2osOmYRIoDXDdN45S3escnBMsUg+bmKljIKebdFKuEdejxzMOqSCPXqYkgMyy67G6DK6KkgqsDZga0biu5b5DQio8oS1u/pgi06mv3+rGMAN3RWlIXNiHSbUA1e7FO4tw6pCaNBuANVettNUyTsNqu8eNbtXXKNQJduV1fnMyofj4DnU75Sw9j10Fc6XToj5lKX2aE+vrn8ufPpPIdzE0bPUaVkthGdnerWOKO/tp4bwb5o1XdbDMmrZnt+TvAPE2wGrjtQdeZHu8lWG1p8Jrcog5vfsvC/F4LfyTatCzg11owQgy/LFTzLaFqZ7Iy4FLwK0KTdJXA8dV8lp0+SDMJVAHJkK0Krqty3pFFNFk7CThsD8SilFLrVKZ5Sl5xX2bBfPODM+fSy+gz2MlCRiDHz9cwFYpB06UBNEyYByBhsAxAxBIf5VzHCVKoV4SBAsyIFtANCHycgOi8nRSP0eLoQsDabYhAQ8M6grNF0xpHTXsgiVJ90ijHbOGMwHuR+uaMYTipLuhNGK6WAZcyhaW8LGBgTgRJn7BrwnJbw+4xzd3pD52B+PniRYM+FG8MuqA4J/72CUVcYfNQRnNQazgMQt1ttvUEO2oYkNxdWPtV0Vg9zBR3qYoVVQkf1rBJQYbqm1LqO6kzL2UZWhSic823dTIFpuyMH5qiIE2asjljWAnJ3AmsPMgkxn7h98D9xsZUz8r0WcBfCjhEIUvXtkf5Xc1RlWrH6Ekf9jaYYvAkei1IbPkBNk59QguG/o+Nb2Vt7Y8yGvAHqSrVJijGQEWjRrigI9JM8Nz4T+lRyYU0kino7cetvxbfGEGZCgb/WNH2WQEU78tAQNw6pCXYRAYYYEmNaK3yfhqufP3JKRVdWqSy7vgU0gXrZ15RkmJti59AlcHs48c//m6oJX8RKCNvqhIHtyBBwsQTOdWaxIEfZAnowmN07yKpfL+GNUVCJ88A5ZmvYHPxVUKv1Avdnh+2DzRJLnoAX5jFxTCoEC0DlIXGeLdL3UbDCUzo4kDVP4Y3bUMHQE/4mJKy1csDO1lrog7XWTMBVP2e3R89MmH9cMK1lnOAiuWHZbrVJFnP8Rtm2TgE/SU1pqq3qJzqQdt2W/D/Az4CnDSMgLKyh8pZx/x7DDQpRnDdvzz8f+XZ0557hAAA', '02_train_change_detector.ipynb': 'H4sIAAAAAAAC/+1923bjRpLge38Fjjxek2oKhTsBeTk7dVG5qqdupyS3Z1vSsnBJSHCBBBsASyXLmtNv87CP87DfsS+77zN/0l+yEZEJIAGCosQub7dPs2xJJJCXyMy4Z2TkzW+UvZClabF3qJz+RlFu4Ic/mZbXCwZP92Z+/jHKruZ7I3qXRPgwdMau4wdMPJyx0o/80odXN7f8UZEt85CJZuH7V4qh/Mf/VU5yP5kn8wvFV4Jlkkb4Mbz05xdMiVjJwjLLz0RXyl7zqa4WZ2maXRXKK11TFj7UcpXBWfP6GnpVAFhllkUsPdsbHirlJVNOsjy8/I5lyoen1Ncz6irJ5h+U0i8+KldJedl09vzpwXHizw6iJI5HAOir5OKypOY/UEcs/0A1AHAWflxkybykEc0jhfl5eq0UZbZY4KNsTt1/8tMEpgf6U57rB0WY5WzUdJdCJWo9XOafWIE9FvA1ZUrJilJhUHtJlUfQWM6KyyyNlHI57++gYCVB4itQdJmWBYwwSJnaN6tffaXoqvLD5bVSVhO48JO8UC79T9B7pgSMVqnsq/yOzz7VLBCQt8dPn42Uq0uWMwCa5TAP/mwBw0gKgGY/BXRJWaQkM6yI/ewfii8n+qj6ZHDg500//txPr4vyIMp9WFeBKzNYNVV5nuU0/Ff/+X/KAhZjXvqpcsU4+EUGWHmJY8J1BKDm5aGyv5/N2f6+ApPFlCzmvebXTW+D4x9eHh+/fP34uyPF0AxzyFd6f7/CVqhMIymUQXGVFMXJq9fms6GqnNC4ffiZZzD1AEg1YAEyzeyIhgcwNz3CVM1LJSkLlsbKbAkrXpT+tbKcF4zNe5cNulIWOaOZh/YJ1XEwSA+FYv/5T/8+Hikc0Y9LP4c18QugriK5wOIxTRqsSZGUFV4hUACBkkTMx+VaFixq+sORqVKD1B4gfZ5FyxBaLLMLmGmWi6niyHtQstkiy2EeiuWC5Z8SaFKpUH1fGRyfPH4vEcEfYPUuFMTdVMWJ15Fy/Y+AhVcZzP5yjtPHomr2KzyCCS1hskuc0pkCYJ3tBQxGyM72aFBne35csvxsr2fixbpQezVfiaCXsAQajvNshuWTvGFUiHWFqjxWFslnqHMJ/fFGABrApZJjel08kbCYffapWUA/XCrEWhwZH8Rh3yr/wz+cnZXsc3nDu7hVJop4UHVwO7050W/h6bfixb+8fX8L3/rKGbfQXk8vfFXnxohAqrAKoQ8YEE8kL/uAGK41HCmLZXEJqIZVgMYZx5t4mTdYAG2yHIhMQqOG8Asil4qOkGZoepTgWgmzeVHmy7DBS+yEozjONzAlQiLgM7DqM7+E1S0Qp5t+sMIfl8A6kR0O5PUgiKG3ZE7rwAHCPuZZiQ+xZoZFhrxnWJ+LeVaxM85MAGJYOui86ZDzlwJmrLwCmu0srjJAfCzKkTJjPgjSkZJnflTgn/Aj8L00Xc6SORHisJfcf2BIj9TiR1iRgpM6zGIv2vysvIN3Ckpv5WfgrPjLgF9PZWyHZ8wPqyXks/uz1MbBwcHKj9zF/j6SL2fi+9BaeJkslP0EP/oNuxWPf8THSDpUAnBUfKHn/mIBy1mJzygpmgfVunEM4VTwrdR64IcfL/JsibyroCXswgioebBI/ZD1Qlp9hJXwowSlRZ6EfgpECmCAHgLIS0AVl0mMnwE7lwtECl2ZIeRpqvzE8gxbylLQO0ZKkJOmwIqCV5xhmTA7yNkFtJhzAV0NIGR8YPv7ADrAJtgRJzJjKA+m+fQal2qkpFlBnE+QapplC0QvHEAhiKZSFMLLLAm5YKrJ1Sd5zdUndQ9bPodfpLjdVwFknuYYfqA/QAEEDRCUjSz/GAOn7VdHvlIeq8pLlBslHwf7DLIjmeFyg2KDalaxEd4Q5kiAxT6zcImzPg0BT0p4O1+mqTyOwHHHltGryFaMWzRRP4Kq2WIZqPwNmybzxRKb3gOx5Rzo+oGhnRjGoe0cao6q64bjGn8QQ6yqgoQvl4UaLIvrOyqanrOmYhKlrLei4ammbtma2VQETp2mNbA5W6TX66uatqf9gWZYucXfYi2zZQljJCPhvH91E1o05ccim8scnx5mxcqjNLu4gMVceV7CWq88vOJKg9QKCeeFX16mSaCIUu98WYdf7bFW4gH7X628BkmySLMS2lMX1/gJiy1kzVcUnC9ni2t8OV+svFsAysIbrBmtvMz9AmRVkq2OGY2TzthqaAEael+NUhggm4qrfgg0gRI4A8Ek6r5+d/y4ebyxCeSFyGPr+k/Fg5FyhGbOsbByRgpxpae1MbSxZVx91sD19Pj3r+hJp2IRJotrdR5xOS0KEzem+Y+m9LlTifcAFJsWKlJyVe8ZfH4FgpcBn8bPwE76sKVp44JlKhqHDZht07GvNu8cmp4Cj/NL05gCXs2W6RQUqzApoBYYqzMWJcsZ2KZNvQrD1RiMLZZXX6EwV0BQfz3bU/ejjHFJRxbOzJ9fK1fAUGEu1f2tGmSfQTBxVitkKGjt5fWWrb1ifny8YGGnOlocc1IT58rp2d4KPvDqzePYB0Eanu2dHyrKV8olKJegK8GKgsrG+R+IuqJA3arpA38JpqKCNcLxaYC9DnE9XoEGlw6qAj88fv/m5ZvvhtuJPj/249jT3fuLvqNVKQYacMbtex+s9zzx56AgChUAdVNRTvGxAHJFWqOcHeRLrmDCWrEgyz6qX1YeWlZoRCZzfxF5SCJm7I3th8pDXtHyzG3koWnonr6dPASBqG8nD79SJnf8U47+5d3R+5evj96cKMdHJyeAjMd3VpCUJAWUcc5x8dNf+k9SLd8+O3qFRuZeHBaJP0M9FaxooED8xsAC+f7gDRjoZN75yntW4DfQYwGU/Nva6EHPSvGNEoNRvsyFilssA9BVw5Y1+OTx039+8vbNEfUIBees1F3en/x9VH8zLfmbrZ3Jvqj3RyfvH798c/QMmjvJl4zABpu15Hb8S5QgBDxDLiPxjefvj47+cDSVoHnuo4UH9bGdQ65kA0mCWYBDjBiNt1d9pZVB2fIlFkZamXePT56+mB6//AOCZ9gOApcDO0BJVZkAYZ4tikPFsUaKbrgKcFwsOJiAVY42CJg6/MVM4suPv/+OMLCZMtFqnCYLLhQ87SBiFzlDDlyS/QJ2BzDkeYmmEvmSArCY+ao35jqIzaab48evj6bvXj1+ejR9/v7x05OXb99Al5pq0iJdIoKARXu2N88EAwQs4G5I8j3yFUSeSAbboGDIHUkEK7o0mncv3p68fQ148PKpNCIwmNmCzSMyGGvz7FGYzUtUyB5x0+3RhT+b+aJ70N5Yzj2Eg3p+EQHkzh6/fH88BRqeHr17+/QFdGjrxnqkeJUJkzBbAC9PfuKW4F+GFa/eHh8T8URg31WEE4RCGPOH+CnOYKXgPSzV2d6PoBf6edSinbfH0x+OXn73AhFB11QNW2KfYeSCWmiVKycmOr4ABYge0EvI2UENC/Uvwfge22QHtNS1nxv9QjIjkLDblfjR43/hk4stu1qLHF4evXmKxXV7/Zwf1X5zIToX5K9MgiRNymtlu0k/+f7N0fTkxfuj4xdvX8nMBszsrGDCiZYFvuilscP7PfUDcidpqi3h1vERsTFNHtDki/7rmzRJIk2UQVuxAjlwQ+Lh9uCm4pS308VNw5hup/7y4gZ0/4FgKsPbKboabnqI//DiVsYR0cF0cZmVGTUhUTI0c4PIdTuNgQn9xOh9h2dDmTS/efV+Tbvzmw6x3k7Z4qbBLgCUsegGZ71VX1qRVyoWmTZewAGWHlW69wSRQCq+yBHKM0ntO0RSbGZ4S8Uzdlzdgn/3Vzx/Vo6FHvmz8nsgB3QJ/oCO06RUiAVmabHW8Sc70z7Q8n8YKR+qWf8ATX1o6wr4WpbdWOQx6PcJWkzLnKuwRfJT7QEn5h9VOoRwfbkgY2N/mZbDtj/vQ4Nt1DfIN/x7vABq8lMaDnAt4t0112afaQfq22qrMCZvIe1+lJnwHiuzrOi4Dj8IJKZ+cHHxw/uubKwlYu3Qb2+itZvsIQRqHuQg/n2RXUEzpfAec48sIB1uaJDLjtyYI8n7mYLuP1JkmdmZrYaIWsM4TsAe5dIaxWgGA4nJ9OKcXOqAHuJAgaGFIDQJjTor0iIs6gfkIP59s5wFMLcwM11NhU8O383DxWKLLLzsNIwkLxCMC7IPFeI2syNEKTrY+eqOQONjC+VZF28aUkcEraQHNe9q+Ei38ctr/zNMjQRoyuYXqOqubuuKDg9BXGqPTG2oKt8X9c4aU55mqR8o3737XgFjreCqcQyDLpWn/BmXnKbGx074g9paH9q0BU5rJZ9uJ3J+Xud6faIq6CDhzoB5AXY8n27kMX113ggLVNEVMKWoKGpooIU+ZzkoXCPln5PFAlYKG3zlL8NLls/9dMHnSczySPkhSQEL8JkEKFSRdHQ/Rp9VVRN3dsCIV56kSL/zynHhzzJYGtwH/MImsa1Fmjt2jF/MJDZtzdrKJDat8fjBJrFpqNbY1DT34SYxVTU0b7ylSVxhTAHM5RO5gyovRvHoW8KzR1xL5m09wi1q0vxTdN/g7ssAMP1HRMw8y0p5u+z927eouKATWA2vosFQXfjExpJYfkhuqMmErA3RM3BOUsKaUtLu6IuX71D9pNYfoVqNMBKWy6rC2+9Pps9evpcLVkNADsCV8yZYpL11c9HYTa2IjZXm1dnHKMkH5LKbZh+7Woe8sY0iaqIsIjVnfjQNi08DPhKEjN6q8Kzlo+v7BHIY4IMGigVwlgH9Hh62NSyYBviP8w0wNLmrlgMweDNSzJHyAkic76e299HpPX8J/A+EETaPYQUgmNksy69V3na7vyTCkfExpFl4yj/xurCu9GEkRnm2d66WWQrTJa8ptSJiBzggE0RZ+OnojmT1JItpguwQO+4MHf+RzKw8+2oGlmY90xwmnHHeG2DBI9KkRaO3apnA1yH6sos87Gm8gVTFXdF5NIBytKSD4XBLWGjE24HCox+6kOhdUHIGuh5Q9gI5UfhxwEcwHDVPqKEh+hGKDQiIvpSJckPgH67iIq1RhTfoWiay4mbvJzR68QPKC8CE27Y/WiDKoIUJBNIQ28KeVdBbZ8Wgi/G4mYtKHyAb7REPNiMjr4LouJwnf1yyleXjFgOuCB+qXdyCKAEFZIAAHVrRraAp8oXcfDNSvlF/BFYyEMAMDw23uG0IjJvpyg3oJ8q+WLgZ8+cwGFWPb79eQ/rIofMZSGHhoEBRAtSDm8pVOIxCfraXb74TAJEjYPBKcw5RF03mMTZAlalGI9If4TpIvb4+eoxuIJzoZuHOT7VzDigoY8VkoI0UA5iIPF3HJ8/6qxVltL5WZZI9QT6EHZBFhjCoFEEAaIwLBSOO6A30Ur/Y0loLIss03Uh7yA75V8pT0GuqVcTploJmRjRqooIqXgKMbjQv/XX7USLcESNCig8K0mUhYgsPyL/F15B4rnDw8JC9ooqGIduDrykyBwzxa0IHJML4QHCKsEgw6xZM2R/gtsVICQBQgP2SfElgMydReTncFzGKQTL382vlA6Jop7a+Uqc3OuaAogD39yuJur8/EtGHuH9JdgaaOnN2VVkioBiUc1A5KfCHHtfRH1XkB/dzFlk7YEyR7bLGUOJqO0wY75bbMwMQaGBeU8RqPkMJR4snL5isvlSjaKvApOvigEjHp27QWxoDdUeHCkbwcJ/oBSsL0pPkoeH3ZjgjeRjhJRgO88qTT81Vk4OODjLLsGFVOcZY1aKKHskptA5M7Rl0hDpKHfIkDBmS8vyR3F8TxQGzWkXC4S45t8pz9sdlAt18Yb090tnYdoL4F9HbSRUee5bzUL2dV7TdrfR2z7Jcezu93TNdw95Kb0c1kJxznEa4yBwp+fyiXxkUkWaNt32kVO72kQiVAqaQ+vOQe4S47x0opInJG9D2+UjRDgzbHvYrglRwIv4+AgPaVrW7iuzvI8gogZEiB5rqjhRdNewhKucEQ7v2Be77TLp1bKykQ6Vua56BbzT0QiQ/sckANF/gYCA9sPlOYBifhHZ3KJQqWIWs7h8Nn3/lgGqswsFHNVR+y1tEw0NM/jo1LUyThWh0n0YN846Qw7fOpPYaG6kPo5LkzEBEV/Qjx5OkjgvuSBxps0cWUU0UNdkUddxhASMs2gF+xSqmtEFATJ4CUSfldDrAeOuR0tH/sMUptTiNcceSolC5I2jyBvBzxLnlhDYJRzJZTEkAkVE2Ii460YY9ijT2qlad0pfKBmlBsqZiD3ioh64+XVNfuLQmlW8LZA4pmVxFX1OJRiyAXRmweE5iY6KIor2l5M3fvmUBQMSq9M2bQFdpFHe3BiIRMa1Z53nEPvc2DCJ6gmTABaAqnM9TeD5ohvZb0QC6E5pJ4Y6D/ro9s5ngKmMzfa2gfoqUDH8ZBkcNcGEkbBn2mXz0Suc4oyMmNOVPk3MViB4EJ6i9IhoJ1H4Z6aBIT5MxMRQ+osFQ+a9rMW+NmfiVclw7rA/xGMpKcK1QnAAU8jGjZ1WE2IbZ4lpyalP0S5r29xNByeiz4ND1rB0Y5HQgHmwQ7+Uxu0ZlE00ofLcblMvyPMvvsMANxBERcDaoZh0NDQ4FTGuWRyyfwBPUESdne3Pm52R5Djc0uyJaDS5b+6uJiSGMxejjYpomHxmZ1HpPFUSt9etU8dLDVtQ2rdy3VU/iPE//8QWMyG8C7Pv7+ZFkFlCQTlS0ActBroHMHCpfK903d00jpwCjQwE/bqaAH883TDOfWeV/8A6GTYNLGIK7fnVXqEhTbQpnI6uK77uQfFsWSyKLlPyiEnGUWdbf9pbYAyChm1ziO7Qd18vV73JG6X296rxX0nWaeJBPaNtJERjpNdmVwlK7D++pZq3aRKp0B6K2b+kwAZ5kup5xYCqbTRxzwINUzSmVu8Y0ama0mkX+YrUSEgynP+7NOm23cE6TcJftu1a43QhX4Rno3jySFLWhKUUcD7DboUpoPCBfBQXZ9JXkuHuK6sr5UE0zFEa32/kwTCd0dct37u/DwHM9spErjtfIXiU04ugsjv+RCTeRMASXc3Gc6lD5gPusU6g8pW1WNhh+wLMiaSIHfdYeimorlnrDmnR4sT8w6Vulx8gGMXOFuLIUG2po4qaslI9pLudiXEA5uFv3n/+L/nCf35e1XE3PMQyzd9q/jOXquc7Y3cZy9VzTcLawXG1DM43xVparrXuO521tuXbxKPDL8LLfNnkM+HXduL0Ez3kIbq23VKXNBoLgtKb181H9hJP0OTGQJyPlZKQ8FVskI3qgi2+rGxUJMtYcaUco8yq50U618z6d9yMpouWAsw4cJ34D3cWCfvTRkButfMLw2NNHMAU9TeGRgMUaNbQ2YVDzhLqeNviIAQuzYkJa2YHep8HSUbCJUumj968IsqIBf6DLwmKWoDYHykka15Em3Mi8QwSMKlC4BY4LPThArzCta/V1zdgR9FE9iKqRUScQUmL1tOKbd984Ck4xuJMjLtds1/hdxCa8jLwfsPgHig7lnHYN4uI8yWe2+xE55tE7/7iKwwLdDvTz9bZbZ8z464VyV1OdmRspIamSgKqf+NbPJ1Inukj8AnRIgvO3SDADY7RiOnW77IHiVFVV0GkOc3QzQGPQ+WEoPp/3t1ZR76T75AFt3R89uN/lVcbqk+/ifAoIitdZtEzZ4JVaZy5onq7ZwJVP0MsH1oWTuCvR66jioidygyTqw10xohPhQlngFEy5HWc7gkfy726vYwVPmQ+Gat3oOneG6KZyaNTNigdNP7CQdVm5WFNiTQ/ClccRs2TzIssHuNUETA3VgYnQ2LhFMlQ/JewKdzt0slzJabjO/VNG3XaPT55t0+zqYhSAeQuxEiDQL9ha7xXp79OIe/kAHtn3t9/dmDs0gDH2xL7VXrVODNlovc2z9t+qF06OIF0zCkDZDWOgHeS7RsBdgZLHT9c0bV1/SBObJo3vVD+kR6Pd4+qyNouV0uG3TQ625pjcYHWxW1TYoRYA6HIZxykT4IEtMq3iY+8GsVqJ7QCU1vFu8O4PUL1UW06ZtNJfCqTKgmKy2/oOmOpNgpohqWXGH6oR+5SEDHS8RzVXWXl5FyxguFBSjSkfCmDIvIjF4MV4+VY1n75pEvU6XnuEMIHTDLVbol8LbNCU5VUsINq+jXtDvFQrx8Ma7wZ1h/7EHqWrLSrWO31kCXO4npVVXfUbJ/fVoNaFzMxI0EPzd2oHlbgVsc4tiduEWrcQuDmqMezrUeUy5GwvTtDvuWVigSgONdO07+9yeMJ3pZuIUnSloesYMwBVBgD3eMkb7Pv7VS6SI9xIz7MrxB46e0kq8GHl5ZSmeY80Z5Fbhge8Slk1VjKVVGdO2Bf2D7i6pZl67P1S/gHbMF19m51tqGiNvQf7ByxVs7yxqW3hH6Cqmj3e2j9QXGZXU0KEQc2zkzlGnYPJvsgZfKJwELH9VyYl2JH0pV+Tfoe5AzChlUhltYIlg+ocWDYnx6/IuyP6oWBIQlyOnH12WJilyxlFoJnIcyQgEYOJ99GWktXxEyRAG/5nRqGhaakWywATHXAfvBjycFS1jsrGBd/KtsD8r5/iF7nCsMcbkWOWG2D7FBQDQo4yAdU1+vRKniiMx3XBEpxC5R7PPIV/N54OY7iGxeIgTwmI8lxNZrjEA96FbOHBOzAZZoBgqCYbI0UbqtyD2uPmH96jJzz3T/iBkXzkJr6BcdweKic3JRqjt73bBU0TxiqwlQWpgUYYzvzF5GzvIvevMUTt0wwUXQ3/+p8n+qaGG9jOWomvgJehJ4Wwuo09/Bk2sEbwdvCukrsb18Sshym1QP08fIwrLcvjbJpfmXjEJJ/wE2uD0eT3iWv/s4oRhdBUhuebukGbQEMlGtjT1L8GZjPoeU/D3BBmjRpsvd90KolTSbU9Tc4bbKh2AUSYCMWpDoZdPyASaX9jw+F5NzISRvf7xobn2aV4AGoD3fB2VKc7aweYzhccFKksjzPFfYrutAra4J4FmDYFs1jwWtBlsZwNPmHwrNZ4eGQQWqvQ4d2rIx0ppzg6gM/PLwCHZAiHZBePz7fUU8aRFlh+HN1fT8HTDwpArMTsqprAatwkJcijk8yq2Ch5ZwToC/dhaVcQcywmmBEGpvCSy49nmOIJk0JJyWUKcV6dQhLUdYd+nqkilwBKpXYLzaeX9YFvUGpE2ihFzpIJD4sCY3ga72J5mWfLC8ycWPkkQeFqJw3gKb/oQH3RjrTcb5Jk7QsNDrhplVPgC2tTLAqYF2jWL6NNgYIyti3nwed7eEXNsB+qTVmOqpmWoXkP16aoqmla1lbaVMn95500OZ2jxKQfT+g06ajrRg0/BiA/JtUJ0857kbph0uR66BTgJ4SndTudY8KjbrKYopjgOcPO80VWTHlXk5aj7bQ5J39OYUD8zHt96J0LTVISO17zOXGcObyfAMNRvlLef/eE7wzXiQY6oOWTV3JuzF75MZ9iDO6MziEAz1yg6sJSIQYW5BjHjQuab5WKMkyVOFiNim9Od3Nnb3PEG9C97uaRojOHuLoyS9KUi4mq1a2tvTgwbd009IcFyR+pPA9WY3H1Br930lJ9UD4ytuBskQdp03Dxa4BbwK0svSPlQyvJ1QdgWrBg9bmtq0s8eSjFVOLhUEoVlWefyAr9UKe1+gCGYJjlUB0j0qs0fe1Tq0fDL83VLMPTLKb9IlyNGMXY0syHcjVeEf5/AFczD3X9UHfVsW161sNOLUpVDW07G5FL5XdIKe/y7ALETzGoEqGtsQCxLB5ou6SccKKSkmL+Kr7HhgoAj5h/+C4JtTCx13rneQ8TXm6DG487dwmSKR3qoh6Euwys33Q6690zosgyoqBJVVoNlznGLfHW0NrptRn4269boKKul1XnwOGbvu78mdybWIGpoKb+GuLUz1o3HHA+3usN/Tk0o1vlZ5HxB+WDcjM7/YZPEn795vxQNWMssqrVthr9hNmfq/qocrZr4+vnev3yMUwDsILnOh22EMXA+rmji3RZXHI/e3+hDRbHV8oLP4+u/Bw0z6ffP3tMZ9X/UcH4B0afB6/fHQ/hydN336tN+vNvCkqtTgdZqyzRJG6g9Ah4aOjDE7mXd9eUeF0J/Tn6wkQOQjxA9HJegs752g8LEWDY5FiHwmhKYiROIuut9YZ/uIx8NSmm/ic/SfEUysoRPSn5IWWluVgs5RVjKbTVTob4sPZm7eO/PUGW3RphG4LqONpJk16Ijp1J1dac7K2E2bSiAGi+I+dWNC5gH1k+IbTsINsKlongVTCVVl5FSY4ZNyfVYWd+lrg+00xnSpsUJ53Khf+JTUFCTj9O9H7NhlIsTKsUC/LwWnJ4sGE47RGgTztBK2JSpX2QeuRZKFFjrQT1gMCEgbYGiUe0sS08Oz6Rkrj0JoAU+xCTKlHnYC1eTKTP3ammTZiiNVXk+Ksyck5OezBhpKyZw1FHfA3PR6uJE1k+4X9W3/HEN9P5FDWVVahgqjnXLyZNqo1OGTZHwppWwhCU9FycoegtRzrdFPRaUBqvVwr2zjzP9jahXIkq/pLdLpXYiJNywC0QVI75LhUXcZPmY5+CfHY2JwzEqFnc+1JueuXebXUQbnAzkADBfTgED7fcHK1SoufDljOj7uvJijIKClrPequotVZThegvBMeWqrg31gJLC42HqeLPQT6076/4smrs2HDHtmN/cTW20QzH2oOM86aiM3bcB6uxnmpaHmijW6ixWNV1te3yUVZWRzsrBOa8GHCyx6S8yPqGPE0AL76SJgIM1mml+VVlMAf7IrgeCGUKynNvIDn+k3CKR8S7ySoIdauG6kZP+5n6OcznZ/Te9RL+2m0N7tSvdzF0+GLJ1i/5jbVzFUsPahhUOqEzagHVqIAUrkmO84l4jIHse1u1WimGrTYbuu9ttbXFQFkFBzw/2nC1uH7/oa3M+H0Gqd93kHe0ftdw9bY3v7o5p9rRFS5zss9W0g03vv2uYoYu/U9oiw0aHOTHUnPaeGC08ZAW8OXggD4SqPEZ4Wx1uvumqdyZetEJQv6Zqkp0sVLqIk+igZ8uLv2Jppqr71N2gXaZTIB3bDoQ9oMGAyg/WElEAw/Rd/pIM6ZVIsYpZ9jqAlcXhOEimeiG1m2QdjC23MsP4mDsROP7i5T3wJn46hJorZsKDhvvCzqOJQlZe12aq2iQA8m3W9CYJTNxQbnfqIhfittKMOgEPdw8W8CnJE19vHHCx3MbjPq98Bei5QxwGrQJnonqBRO3K7STnEn4nGNyH3GFkZQmoD6v0uRxxXGjcSUuBJpnSXFdXdCDz/FkwhLPIMC49+sTCfuSerT+NguRRoJvHyBMlz6eo8E5WEkVJraf12wbfKdW+TVZlWaMjlGsyar+FrOEzLMrJcrwfqYyW4aXrUrKoE7yNRyB7VelvyCaa3SfL53semz6LDbZL6NcgLy2DccZP1i5oIqGqz9UuTA0VXdd27IfrlxQ1bFrb3fOgiLeGg2jjreCx5vVbWC+HxfllGxLzmRp55flQVYwrvsPQfT1oRWoMsegYrCiY2XdrHpFzvbA2l/mfnhNZ6YkgEXg5fQtSqk0bYqdj/qaeVdl8l/XjpB2Urn+ht4zJM8NrVSF+puoROKGRiTB29fKy+z7DQ38jmcNfonifbWV2853spN5AwrtwTYZCTXVbvkVhiLJjjlUywwPFqO1tGW6Hc9wzeghR9UeS7KFXC+07rTVCgKWZzYSTi3FTylLKPrvxc4tss/6vNq3yv5+rZ8gr4Y5BT2lfVJNVlj296kBusSKuNNaTvtCVU6W8zvSPPbf6kUJXnoTQI6EK45FXc79hblrEBn22NB+Ge6KDMvTXevB3JVXtPUH70BYlmrqju3oD+euVNV2MFnMFtyVVM3+zVWVUqLR2c9GSA42+AqQ03ZNMdF8/RkDgumviAdW8RbKwT3yBYqYG5GmLborFch7HtRanTapcTsRZ5NyJkIAxMk50uuRACl6hHIGivZ7o+SwwXuk+MPgmSrjGzR1uP50mxRi0sQoY3X58NJyXvxxifvQA21tjj7uxp5n04scU/kdrtu8yILWktRTW7LFQD4rzCO3YcWasmLZbjGorecIM05NlckPv6gV0EM1XCwH6w7q1XXEqKt4pKZ2kGXpYE1GQHE5Dh2aJgiGo9YznhZwM4oRly2oCb68koRZtyMnZDHyPiSJEe6/NHx6xZ7k165cJJ8oH7Noew2WUeITWqx/7GPK+Ktc4ME1KvpfCOKhinv2VeRWBynlwv+6sTSeNBr86z3brg+ab1RsFmA8ossFPvwWQMLTQ5u0mHaV+doqstZiKPtNRfEF+7u7hUpj6QC5rtLt5vg7jkz4SZwCXeVjPXF0snwXC0+sRlM1HShP1YxzgOkUM29ioFwnfNUeKZ4zUuwhFdJUz8U6nnfeBo2jO3fcYfj+c9KSTgUVrAIP1DBsumoAOxdJcCbNo74BQEdNr6fyajVuONzJ7dyEUN1j0DfB/ud2o6ctZBtJWDRqoce5cDDVLrzxiG4fRf2PobdG7TiNatcOAHIPn87ZnoiOUwZ0A8OaxuqpaZp8arY8QyKf3uCmLnqoGvGKT67tFOpX5npqVA6w33eujqDE2St9rHUprXUnrY9RJRuAYxe3DjbQhnzyadgy0xrEvYel1rEXDiuO3wfOiC7P6OESsCzlEu87blrrLM/Glhupsp6hDNWT2obZMgQqDMaObTzgOrH9/feUKBqMiAFlp9K4DKNrepb8Mm5fZKE/sXBDfySyILlyynlbN6q0aXyPtU5wL+3KAscQuamXc7H15XiikcPVy815pm7M15bFMUnRAEN8NXXs8nhW7j81AeI5OaJS1o78rB1h0NZHtiiVPCkonKrIlBldxcvhv8qWKb+3F3TwBb9PuOMswlu44VtEN6zJMMLYASCv6RQeEzGpyltuEUGb0QEC3qSe51lCoMlVZUFwaGgZmrVrvUJTHa3XrHtK+TOru3Oa2wFwimnpUn8BK0G3BQz6bwk47AwJOnY47vJvzliRbyhuUdSQ5xPFxDDyXRkz+As4AF8LOilVX1kIaCYbijQHA5xAvCWgRIctTahj3tklnRqnq7X81M9nhKTU3rpgcCXK+YYr4Q1YbebXiMIoVb+udDZui6+9JL09R7Vr1afUjsKkx9B/Go775z/9O40KZSf30UpqJawKBlkfdC5d58abvIkrWzFhShc0ZDzASW+8vR3GNMBBDckbm17Xt1A913sHtr9/DDRfxw8OVVifk4b2eQ5dKoFZHGneDVJl19CBqyv/8b9xWjVHJOWt0Eh2YEBBYywKmoYy+CAnuH8kEtfjduEH4CbHnCnhNfEFS1lItycLT0pS8tNxFb12juTBoImW6Sx0s7lPjhW8JxHmJcax4MZ6wTlCn/ea74dL5It9Ythdc6k0rvnX3ZTZMqcvcCeA6qRgrEBfn8QVyti2qrzjUpBSU/IFIy8/ygIcLtXJfdqMIAjxspsF2h+lj9ktiyaMXyKa6wWF2fvIMm1lhjcCZCn5SLa8p9k3XYu59gN39V+qyu8TTEyGOyLSKZ11hBZjTtcONYvNF2LJOD8jnrCYXiUlDyWrdL0v7Hly7IiFRhj/Ip4ndOYYmo5B8g/zPJEXyBtv4XmyVdewDM/bwvOEVU1H287zxBdOTjVaa0ftM0hRMpvQfrts+lOONrqsodUKHsXBzPmDiBUhw6wtFzw6oOeuBlRNoQVq6fTQosxI/MuBdXhOZzeLLsqlzK9xTqIsfsINjDNQDEkFP6xOL3XgO03O+eGl6rATOQvwFoKOrkg5f+oJ4d9qgXH6TYXbGNBJ2mafm6c+s4VDOd90sqmdWQDLtI+onraAk/0Rdea4To/1gVb+Z1vfu8FCS3PC+zOZ407mXmXQTfo5rK+sz4iHtrI5lVl+fUhnqegOZOGSp9uRe44EgIgXAfvk0095Mj6RiZpEHjya0/W8/NzRf/vCofhjLzDN0Ppl2BFQuO0auvlgdoQVx7a9FTsam5qlbceOxoaB1/FuwY6aHLRIx0nfgceaHIfkpJCpk7vFKNxcIjRSSaekkvKkjsTP7kFI6w5fNkCe9wX3ndyl8oojl00Tw9tvUUG9kI5cclV8VnMvaQDEZ77eOgDQt2PNcuzxw1SF36mKsGeUa6QnKdc/z8DhC7WJa8x0F3h1cZj5hS/qigzdNqLgF6Ozse0Z3jZ0BmQmk8u96cyzNFPTt6Izz/Qsz92KzoQ2P40TOpMvx0JLiv5ZKwZvDlrr5L7uFPmMwHsGBhhp9Fm1x1lf8FUbTNXDGGrj3RSgTgeMNPKoLzgeVYYC87VcqjlL8e999sRGdHuXBFxOysdNd0eBSxeYrt5rUVfi5vG4M6ZtgQpSWpVuGRHJg6WqhCzdIg1bwFJ9+aFWOm4SY1H3TVaslaI8EPFQ6TkRebYHFvhPjPYF7j5TCTBimneE7mjleKY49v1IBMPhbNyV+YtC+IXrA0uvDS0X4XVUqDcoe6U8P3+D5bnnjnP0+0RY45nx4Up7kvf2sMHylWLP9X8S/kze7UavZtv73tt1axhIhPeoI3ZP2pXoYX/5ZmKwWvNNKig5RKs77Lr7FZ08EpjqWWIzKt1q13fZltRamM1DWKhTOYJZbmQ4qsrjXsfFHOZgyrc8OtHHFRcrs5U2qk0SHnfUf3jqqL4HWBJlypx8Lhg+KNLIJkVXEa2dIT1cpZGbOMtrD9Lcazl60JLj+FqPNkZ8+0ThdCdWZYmt0nfJyfvkWV8ZCQ0GstSoxs1JSf2xoHBf9SoH5WmKt/YO8JEaLWeLYlAV5isxLycto/Ir5ZjHrgTXq97nQx7NKVIfcKcRXW1FF6UGeLUAOvakoBcqUnnSVhEEXZADkfpm0p76oYpG7BSDIDH+rGYqI8WvzVqORNtpY4Yf2lqsmffXxtouuZ45/6Dwo1oy1pLUhQmDOt8UMjZJ7smVFN6SX/JljGofua4LRhF07duecnaA78iOm5OKqFz5IKGhX/QGpmirD9DhGiU8e+3vjt++UQQ10105tHgS4yhx6dGB1PEc0lpWKyyKVFfE8vVcE2OlmKTCzjAYLClBnwYteyAl3xiJxOMRpeAY9l9Ctr//u2V0UYWtRhSgkxEcVWCZSvkv6BDk3cEHarvdk6U4IdgOmIXG4ZEYILb9dO3aFfwOcO6gDOvQMH7xLprEnR4fc12kHgzl86hvJkhpJN+x6sq0Vm7Y2RLAA9UsqTOGoMK/mnS4dakaJWUn0xy726/jl6/3KeBZpJxp3SMwotC9Zb6AwXSg59YIAX+ViauEyDUuFoD5dCh7RvtKzdxUl6GxK+E9D6rLzuZhukRy7vTzPMsvxOXqV9XN1MX1HP7AwIWJJ86yluj5xO7fZO0QcbqbCb7D9LIgTzBcaRnh0l7BZLXuI+GnZH0FLMaC8ZO3mIacqXRzeGNf4Z5XycWQsh8xvIPumkX7ol4iLmxELKIplViB5IP9jYKWQdeA2vvIcsxisWBhZT/tRUkBGun1FGM+kWu9A7kIK1NxrRSWbglEhK8W9Eq8qMrzh2QHElera0zxKsm6GzQDeVZvYmeNRVe1k8itK3vAAgpu5pmS5YNcZQrShs3Fyz11cV3xV2BYFT9GifTo88EdAFcPA1g/6AsUzc8UUpmvFFlcE3UU0xSstlwCtZqiBtQ9U9Xhf6eZjKskwqv96mkgPhTy6zQ/zSP1xyW0BeouL3eABiX7LXL6ZoLomWTuooUfxKFrRxazdNcMxmboxIZvsGAchea4KapwQ0ekAED4/kn0dyDgehT4Bats03bxqTQsXTVUrVusms9XdIqFTjA3RTpAw5Ppw2DpVFgPTV3wLnigECZTXvU7SO8eAJlU/k7AqFwbrt/DI6kI+S2nyENYD2D8LV1iu+YdnjBZeRWQg73ncQm25MpjwQBWnseA770Pp6CJXq28waiVKe4sr3mzRFWd62F3lLijaXwNJsia2rzp/ncX/qL/Rb6uM7ylBuQyowEVG8rcOaq6VC/o/FqZlcc/ggxO4uu1eFG978eMlMWrVUBDBd2m5/Hn6Rog8BVdd7P6JpmvrQSv+itlAfp/wEIr170CTSARrr3O+150pvNbfehSvZh+Xv9qFd0XfoQqwsrzvHecZbaKUp9AH+HRYCuv5CkRD2/539vKMag5nh+FXuRafmSNfcMbW/bYM5xQd9wgcs0dW9+x9R1b37H1HVv/VbF1XYvNsWmarj72LS/WXEPzfMsCDT7yPT20HsTW6R7gLC3uw9rt9az9OXoSq1wymzk8pdJPKT1os+V0T+a/AvGdAsC+SwDcAfVfKAf6gFwrC+w7ZEEFXUcaBH4+Lcpr3nuxDEMoIr3GKJo8WQga2et/My2zLC2TVfTkSQSw4st3/31Km1lT37fCwA/hP9u1Ajf0Ql2Pg7Ht+HZsetV96hU7gLqW42q2bYOca9E8vNHkR1meVH4i7PASvv+E7g95JeqBNuCMHRuvBPBd39CsWLf90I6C0ACzNWDQsSXVJmcsAmSatmk7pqqtoSvD1MJxGOih748ty/G92NRNN2ShBVRljrWdurRTl3bq0k5d2qlLvyp1yXDc2MaLRICxWwELAse0PCcc23bogqxwdmx9x9Z3bH3H1nds/dfF1l2wP2xLs41IsyLN8A3L8HU31DU7iGLX21nBOyv4i1vBpq+b1lh3NCd0rcgbe65n+1oEmkWAr6L/31Zw5Aa2aeIv3zJ05o49yzUt07EN2zECr88KrgFaQ1embsHQxjEYwrHlGLqrBQGzTW9sG+aYxcZfg65enLx+9ashp1Vg/1aoCCHrUNCXJ5EodE3Pdt2xyQJr7Id+5MQh8yLbDMaayWQSobDlS4rexjb+/Kf/eTe6e7Fjj2PDMJlrWHYU+1YY+76r287Ysj3D70F3jkhq4ceMX9lSHP75T/+GN2/MizJfhhipAw8wCARf6Jr29d46wthE+zs7YmdH7OyInR2xsyN+TXaEFejeeOyPgyBwrMjSPd8OxqFpeczTAuDufw1951kje49RDH4BJv8FFZy7oftbYPkE1npNp6aUvXWy3g48LwjD2IiNwHJCw7MC0/VdLbQ9UGlibacE75Tgu5TgjVuJ2yvBG/nVfZVgPFyCmUK5BhzAbNPjtSTBbObrsa1HpmPFYeyB/quHBnN032Ixc3f+lp2/5cvT0aa9m66/BWzBX8bVEjDbH4/1yNXGQHSB74UG2HyGEwZOGOth0EN2BMsaaoIhOY5rOUGAfktXCxwPGARjjuebRuQ4O63j71Dr2BjU8ldAiooT/A1ixB2g/a2iAzJQShe66g14EKJs8vvuEGWHKFjYtaLADmwj8j3HAovGG3u6FvtOPHZ8LQ79v4od8yT7/OuxY1aA/ZuxYwCyLtJknxv1THoeXiZplLN5fSqcH9KudZuN1u6or9bGMOTeWq4D5g6wK9tnnuVYhm9HbqyxWI91A5Qfr8Jk5fxuzZAx32RR6Pl6FFu6HgRjRwvd8Xgc2qEXjf21FGHbgatpPgu80LLAZArD0DfHemxahhFpbKd4/R0qXhuRcufu2bl77mRGzjjSXWaOHd3yLdsdu7oxDpAteXHA4r/A3bPRUOxx9/z5T/9mmar1+skIP6mG8frJowI+3uHd8aAxLzCAmUeWFem653hmbPh6aFgxWB/WbnNzt7m529zcbW7uNjd/VZub3thlvudGtqFplu/rvs10EFSmw6zQjfxdzMqOre/Y+o6t79j6r4ytbwqE3Dkx/v6cGL5tGWONxZGvuVYcjIM4cFkQ2JbnRb7j6juk+HtEik3H9ncK4E4B3CmAOwVwpwD+mhRAn43NONKDGJQ9y7SCQAvwZIppGaYbjsfubl93t6/7ahqxQLdDI3SDMAbM0Dw9Ng3D900tsNzA8/p3aDcGevbWYroTOg6zYn+sWV5seSZ0EI6jCHqPnVC7777uxpyFaygC8D92oBqzxp4VO6YbBXrsB+PY161Yc+PdFt5uC+8uvNvoKd1+C29jyMGaLTxHdV8/gQ+P6i98P8+2N+/nbQxX3UWI7SLESIZE5tgzx6ET68AoQ8t3NQ945th0QhbbkbHzGvwdeg0iwwwtm41BlvpWaJteZMRWDILZjuwgsJyderlTL19NN2aK6FUUN+Zt6a21Ub27p3oZeWbEjDgOfeaCYhm4hsFsG1r1bd3xLWctRWyqt/Oj7fxoOz/azo+286P9mvxoG5PV7Nj6jq3v2PqOre/Y+q+KrW9yfO+cwTtn8F3O4E2nIbZ3Bm/0t/U4g2Hy4uSCbmvdlKRu4zbMDvN3mH/X9tummyC3x/yN8Wn92yC67TyCH/h0qmmHmvZf0vJb+oCbIbqu6h+fPCrO19LDppNZOwV/p+DvFPydgr9T8H9NCv7G0/87tr5j6zu2vmPrO7b+a2Drv5GY+55gJtOZ/yMFtxjd58mcnlNqPawKP1h3bx7EWT7zEVBL/l7XsH9z+/8ApAGdF9saAQA='}  # the two project notebooks, gzip + base64

for name, blob in NOTEBOOKS.items():
    (WORKDIR / name).write_bytes(gzip.decompress(base64.b64decode(blob)))
DATA_NB, TRAIN_NB = sorted(NOTEBOOKS)
print(DATA_NB, TRAIN_NB, sep="\n")


def run_notebook(src, dst, settings=None):
    """Execute a notebook (optionally with changed settings) and save the executed copy."""
    nb = nbformat.read(WORKDIR / src, as_version=4)
    for cell in nb.cells:
        if cell.cell_type == "code":
            for key, value in (settings or {}).items():
                cell.source = re.sub(rf"^{key} = .*$", f"{key} = {value!r}", cell.source, flags=re.M)
    start = time.time()
    try:
        NotebookClient(nb, timeout=None, kernel_name="python3",
                       resources={"metadata": {"path": str(WORKDIR)}}).execute()
    finally:
        nbformat.write(nb, WORKDIR / dst)
    print(f"{dst}: {(time.time() - start) / 60:.1f} min")

## Notebook 1: data (about 10 minutes)

In [ ]:
if (WORKDIR / "data/chips/chips.csv").exists():
    print("Chips already there, skipping notebook 1")
else:
    run_notebook(DATA_NB, DATA_NB)

## Notebook 2: one training run per seed

Each run prints one line when it is done. On a T4, expect very roughly 10–20 minutes per seed.

In [ ]:
seed_notebooks = WORKDIR / "outputs/runs/seed_notebooks"
seed_notebooks.mkdir(parents=True, exist_ok=True)
settings = {"MAX_EPOCHS": MAX_EPOCHS, "PATIENCE": PATIENCE, "PAIRS_PER_EPOCH": PAIRS_PER_EPOCH}

for seed in SEEDS:
    executed = f"outputs/runs/seed_notebooks/{Path(TRAIN_NB).stem}_seed{seed}.ipynb"
    run_notebook(TRAIN_NB, executed, settings | {"SEED": seed})
    shutil.copy(WORKDIR / "outputs/figures/02_learning_curves.png", WORKDIR / f"outputs/figures/02_learning_curves_seed{seed}.png")
    last = pd.read_csv(WORKDIR / "outputs/runs/results.csv").iloc[-1]
    print(f"  seed {seed}: {last['epochs']} epochs, val F1 {last['val F1']:.3f}, "
          f"test F1 {last['F1']:.3f}, IoU {last['IoU']:.3f}, threshold {last['threshold']}")

## Choose by validation F1, report the spread

In [ ]:
results = pd.read_csv(WORKDIR / "outputs/runs/results.csv")
results = results[(results["max epochs"] == MAX_EPOCHS) & (results["pairs/epoch"] == PAIRS_PER_EPOCH)]
display(results.drop(columns="checkpoint"))
print(results[["val F1", "F1", "IoU"]].agg(["mean", "std"]).round(3))

best = results.loc[results["val F1"].idxmax()]
print(f"\nSelected by validation F1: seed {best['seed']}")

# notebook 3 uses outputs/runs/selected_model.json; point it to the selected run (same threshold and normalization)
selected = json.loads((WORKDIR / "outputs/runs/selected_model.json").read_text())
executed_best = nbformat.read(seed_notebooks / f"{Path(TRAIN_NB).stem}_seed{best['seed']}.ipynb", as_version=4)
selected["experiment"] = Path(best["checkpoint"]).parent.name
selected["checkpoint"] = best["checkpoint"]
selected["threshold"] = float(best["threshold"])
(WORKDIR / "outputs/runs/selected_model.json").write_text(json.dumps(selected, indent=2))
nbformat.write(executed_best, WORKDIR / "notebooks" / TRAIN_NB)  # the executed notebook of the selected seed
_ = shutil.copy(WORKDIR / f"outputs/figures/02_learning_curves_seed{best['seed']}.png", WORKDIR / "outputs/figures/02_learning_curves.png")

## Pack and download

The optimizer state is removed from the checkpoints: it is only needed to continue training, not to
predict, and it doubles the file size.

In [ ]:
for ckpt_path in (WORKDIR / "outputs/runs/checkpoints").rglob("*.ckpt"):
    ckpt = torch.load(ckpt_path, map_location="cpu", weights_only=False)
    for key in ("optimizer_states", "lr_schedulers"):
        ckpt.pop(key, None)
    torch.save(ckpt, ckpt_path)

pack = WORKDIR / "colab_runs"
shutil.rmtree(pack, ignore_errors=True)
for item in ["outputs/runs", "data/chips/chips.csv", f"notebooks/{TRAIN_NB}"] \
        + [f"outputs/figures/02_learning_curves_seed{s}.png" for s in SEEDS] + ["outputs/figures/02_learning_curves.png"]:
    source, target = WORKDIR / item, pack / item
    target.parent.mkdir(parents=True, exist_ok=True)
    (shutil.copytree if source.is_dir() else shutil.copy)(source, target)
archive = shutil.make_archive(str(WORKDIR / "colab_runs"), "zip", pack)
print(f"{archive}: {Path(archive).stat().st_size / 1e6:.0f} MB")

if IN_COLAB:
    from google.colab import files
    files.download(archive)